# 1. Configuration

Every world and system in TidalPy is described by a TOML configuration file. This demo covers where those files live, what they contain, and how to load one, read it as a Python dictionary, edit it in a live session, and build a world from it.

## Where Configurations Live

TidalPy ships a pack of example worlds. `available_worlds()` lists the ones it can find by name and `get_worlds_x_dir()` gives the per-user directory they are installed into. The same directory holds the multi-world system configs, which `available_systems()` lists separately. Every config carries a `schema_version` that tells TidalPy how to read it.

In [2]:
from TidalPy.structures_x.configs import (
    available_worlds,
    available_systems,
    get_worlds_x_dir,
    resolve_world_path,
    load_toml,
    merge_with_defaults,
    validate_world_config,
    build_world,
    SCHEMA_VERSION,
)

print("current schema version:", SCHEMA_VERSION)
print("worlds directory      :", get_worlds_x_dir())
print("worlds available      :", available_worlds())
print("systems available     :", available_systems())

current schema version: 0.2.0
worlds directory      : C:\Users\jrenaud\OneDrive - NASA\Documents\TidalPy\0.8.X\Worlds_x
worlds available      : ['charon', 'earth_prem', 'earth_simple', 'europa', 'io', 'jupiter', 'jupiter_simple', 'luna', 'mercury', 'neptune', 'pluto', 'sol', 'trappist1', 'trappist1b', 'trappist1c', 'trappist1d', 'trappist1e', 'trappist1f', 'trappist1g', 'trappist1h', 'triton']
systems available     : ['sol_system']


## Config File Contents

`resolve_world_path` turns a world name into the path of its TOML file. The raw text of the two-layer `earth_simple` world holds a name, bulk properties (radius, mass, spin), and one table per layer. Each layer names a `class` (which layer type to build) and a material `type`. The detailed material defaults (equation of state, rheology, viscosity, and so on) are filled in from TidalPy's material library at build time.

In [3]:
from pathlib import Path

earth_path = resolve_world_path("earth_simple")
print(earth_path)
print()
print(Path(earth_path).read_text())

C:\Users\jrenaud\OneDrive - NASA\Documents\TidalPy\0.8.X\Worlds_x\earth_simple.toml

# Earth-Simple: a three-layer Earth for the structures_x world builder, and the worked example of its
# schema. A solid inner core and a static liquid outer core under a tidally active rocky mantle, with the
# published mass, moment of inertia, and degree-2 Love number reproduced; earth_prem is the seismic
# profile for work that needs one.
#
# Each layer names a `class` (which layer class to build) and a material `type`; the per-material
# parameter defaults (EOS, rheology, viscosity, melt, cooling, radiogenics) are pulled from the matching
# [layers.<type>] block of TidalPy_Configs_x.toml. Any of those can be overridden here by adding the
# corresponding key or [..] sub-table, as the viscosity laws below are. Layers are built inner-to-outer:
# the inner radius is derived from the previous layer (0 for the innermost), and each layer's outer
# radius is set by exactly one of radius_outer_m, radius_frac

## Loading a Config as a Dictionary

`load_toml` reads a TOML file by path into a nested dictionary. This is the form you edit in a live session.

In [4]:
import pprint

config = load_toml(earth_path)
pprint.pp(config)

{'schema_version': '0.2.0',
 'name': 'Earth-Simple',
 'type': 'terrestrial',
 'radius_m': 6371000.0,
 'mass_kg': 5.972e+24,
 'spin_frequency_rad_s': 7.292e-05,
 'eos_solver': {'solve_temperature': False},
 'layers': {'inner_core': {'class': 'solidliquid',
                           'type': 'iron',
                           'layer_index': 0,
                           'radius_outer_m': 1221500.0,
                           'is_tidal': True,
                           'tidal_scale': 0.0,
                           'temperature_k': 5500.0,
                           'material': {'model': 'constant',
                                        'reference_density_kg_m3': 12900.0,
                                        'shear_viscosity': {'model': 'reference',
                                                            'reference_viscosity_pas': 1e+20,
                                                            'reference_temperature_k': 5500.0,
                                                

## Building a World from a Name or a Dictionary

`build_world` accepts either a world name, which it resolves and loads, or a config dictionary. Both return the same kind of world object.

In [5]:
world_from_name = build_world("earth_simple")
world_from_dict = build_world(config)

for w in (world_from_name, world_from_dict):
    print(f"{w.name:14} type={w.world_type:12} R={w.radius:.3e} m  M={w.mass:.3e} kg  layers={w.num_layers}")

Earth-Simple   type=terrestrial  R=6.371e+06 m  M=5.972e+24 kg  layers=3
Earth-Simple   type=terrestrial  R=6.371e+06 m  M=5.972e+24 kg  layers=3


## Editing a Config in a Live Session

The config is a dictionary, so it can be changed in memory and built into a new world without touching any file on disk. The edit below makes a heavier planet with a larger iron core. `validate_world_config` checks the edited dictionary against the schema before the build, so mistakes are caught early.

In [8]:
import copy

edited = copy.deepcopy(config)
edited["name"] = "Earth-Heavy"
edited["mass_kg"] = 7.0e24  # heavier world
edited["layers"]["outer_core"]["radius_outer_m"] = 4.0e6  # larger core

validate_world_config(edited)  # raises if the edit broke the schema
heavy = build_world(edited)

print(f"{'world':14} {'mass (kg)':>12} {'core R (m)':>12} {'mean rho (kg/m^3)':>18}")
for w, cfg in ((world_from_name, config), (heavy, edited)):
    print(f"{w.name:14} {w.mass:12.3e} {cfg['layers']['outer_core']['radius_outer_m']:12.3e} {w.calc_mean_density():18.1f}")

world             mass (kg)   core R (m)  mean rho (kg/m^3)
Earth-Simple      5.972e+24    3.480e+06             5513.3
Earth-Heavy       7.000e+24    4.000e+06             6462.3


## Applying Material Defaults

`build_world` merges the compact config with TidalPy's material defaults. `merge_with_defaults` runs that merge on its own, for inspecting or adjusting the merged form directly.

In [11]:
merged = merge_with_defaults(config)
print("top-level keys after merge:", list(merged.keys()))
print("mantle layer:")
pprint.pp(merged["layers"]["mantle"])

top-level keys after merge: ['schema_version', 'name', 'type', 'radius_m', 'mass_kg', 'spin_frequency_rad_s', 'eos_solver', 'layers']
mantle layer:
{'class': 'solidliquid',
 'type': 'mantle_rock',
 'layer_index': 2,
 'radius_fraction': 1.0,
 'is_tidal': True,
 'tidal_scale': 1.0,
 'temperature_k': 1600.0,
 'material': {'model': 'bm',
              'shear_modulus_static_pa': 221100000000.0,
              'reference_density_kg_m3': 3516.322,
              'reference_bulk_modulus_pa': 128000000000.0,
              'bulk_modulus_derivative': 4.2,
              'shear_viscosity': {'model': 'reference',
                                  'reference_viscosity_pas': 1e+21,
                                  'reference_temperature_k': 1600.0,
                                  'molar_activation_energy_j_mol': 300000.0,
                                  'molar_activation_volume_m3_mol': 0.0}}}
